# Lesson 6.3: Errors and Power

Simulate thousands of experiments to see the Type I error rate when nothing is going on and the power when a real 6-point effect exists, then check with a power calculator.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A school wonders whether a new teaching method raises exam scores. Past exams have a standard deviation of 15 points, and the method might add about 6 points. The plan is to compare two groups of 36 students with a t-test. If the method really works, how likely is this study to notice it, and how many students per group would we need to be fairly sure?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Plan the experiment

H0: the two groups have the same average score. H1: they differ. The method may add 6 points (SD 15), and we plan 36 students per group with a t-test at alpha = 0.05. The effect size d is the gap in SDs: 6 / 15 = 0.4.

In [1]:
import numpy as np  # Load NumPy for random numbers and arrays, under the short name np.
from scipy import stats  # Load SciPy's statistics tools; we need its t-test.

rng = np.random.default_rng(42)  # Create a seeded random number generator, so every run gives the same simulated experiments.
n, sd, diff, alpha, reps = 36, 15, 6, 0.05, 10_000  # Store the plan in one line: 36 per group, SD 15, a real gap of 6 points, alpha 0.05, and 10,000 pretend experiments.

d = diff / sd  # Effect size d: the gap measured in standard deviations. It lets us compare effects on any scale.
print(f'effect size d = {d:.2f}')  # Print d, to rounded 2 decimals, as a check of the plan.

effect size d = 0.40


**What this means.** d = 0.40 means the method moves scores by 0.4 of a standard deviation, a small-to-medium effect. A study plan always starts from a guess like this: you cannot talk about power without saying how big an effect you hope to find.

## Step 2: Simulate when the null is true

We run 10,000 pretend experiments where both groups come from the same population. The share of experiments that still reject H0 is the Type I error rate, and it should be close to alpha.

In [2]:
def reject_rate(true_diff):  # Define a reusable function that simulates 10,000 experiments for a given true gap and returns how often the t-test rejects.
    a = rng.normal(0, sd, size=(reps, n))  # Make 10,000 rows of 36 scores for group A (mean 0). Each row is one pretend experiment.
    b = rng.normal(true_diff, sd, size=(reps, n))  # Make 10,000 rows of 36 scores for group B, shifted up by the true gap.
    p = stats.ttest_ind(a, b, axis=1).pvalue   # one t-test per row  # Run one t-test per row (axis=1) and keep all the p-values.
    return (p <= alpha).mean()  # The share of p-values at or below alpha is the share of experiments that reject H0.

type1 = reject_rate(0)  # Run it with a true gap of 0, so H0 is really true.
print(f'Type I error rate (no real effect) = {type1:.3f}')  # Print the rejection rate, which is the Type I error rate (false alarms).

Type I error rate (no real effect) = 0.049


**What this means.** With no real effect, 4.9% of the 10,000 experiments still gave p <= 0.05. That is the Type I error rate, and it is close to alpha = 0.05, as theory says. About 1 in 20 studies of a useless method will look significant by pure luck.

## Step 3: Simulate when a real effect exists

Now the second group truly scores 6 points higher. The share of experiments that reject H0 is the power. With only 36 per group the test misses the effect more often than it finds it.

In [3]:
power_sim = reject_rate(diff)  # Run the same function with a true 6-point gap: now H0 is false, so the rejection rate is the power.

print(f'simulated power = {power_sim:.3f}')  # Print the power: the share of experiments that correctly found the real effect.
print(f'simulated Type II error (beta) = {1 - power_sim:.3f}')  # Print beta, the Type II error rate: the share that missed it. It is 1 minus power.

simulated power = 0.387
simulated Type II error (beta) = 0.613


**What this means.** When the method truly adds 6 points, only 38.7% of the experiments detect it (power = 0.387). The other 61.3% miss it (Type II error). So with 36 students per group, a real 6-point gain is missed more often than it is found.

## Step 4: Check with statsmodels and plan the sample size

TTestIndPower does the calculation exactly, without simulation. power() gives the power for n = 36 per group, and solve_power() finds the group size needed for 80% power.

In [4]:
from statsmodels.stats.power import TTestIndPower  # Load statsmodels' power calculator for two-sample t-tests.

analysis = TTestIndPower()  # Create the calculator object. It can answer power or sample-size questions without simulation.
power_exact = analysis.power(effect_size=d, nobs1=n, alpha=alpha)  # power() gives the exact chance of rejecting H0 for our effect size, 36 per group and alpha.
n_needed = analysis.solve_power(effect_size=d, power=0.80, alpha=alpha)  # solve_power() runs the calculation backwards: how many per group do we need for 80% power?

print(f'exact power with {n} per group = {power_exact:.3f}')  # Print the exact power for the planned study.
print(f'n per group for 80% power = {n_needed:.2f}, so use {int(np.ceil(n_needed))}')  # Print the needed group size; sample sizes must be whole people, so we round up.

exact power with 36 per group = 0.388
n per group for 80% power = 99.08, so use 100


**What this means.** The exact power is 0.388, matching the simulated 0.387 (the small difference is random noise). To reach the usual target of 80% power we need 99.08 students per group, so we recruit 100 per group. Power depends on effect size, spread, alpha and sample size, and the sample size is the lever we control.

## So, what is the answer?

With 36 students per group the study would detect a real 6-point gain only about 39% of the time (exact power 0.388), so a non-significant result would prove little. To have an 80% chance of detecting it we need about 100 per group. Caution: these numbers assume the effect really is 6 points with SD 15, and real effects are often smaller than hoped.

## Now you try

A website team wants to test whether a new layout lengthens the average visit by 3 minutes. Past visits have a standard deviation of about 10 minutes. They plan 40 visitors per group and a t-test at alpha = 0.05. Is the plan strong enough? Find the effect size, the power, the group size needed for 80% power and a quick simulation check.

**Data:** diff, sd, n, alpha = 3, 10, 40, 0.05   # minutes gained, spread, group size, significance level

1. Compute the effect size d.
2. Find the exact power for 40 per group.
3. Find the group size needed for 80% power (round up).
4. Simulate 5,000 experiments with a real 3-minute gain and compare the rejection rate with the exact power.

*Hint:* Use TTestIndPower from statsmodels: .power(...) for the first question and .solve_power(...) for the second. The simulation looks like the lab's reject_rate function.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.power import TTestIndPower

diff, sd, n, alpha = 3, 10, 40, 0.05
d = diff / sd
tool = TTestIndPower()
print(f'effect size d = {d:.2f}')
print(f'power with {n} per group = {tool.power(d, nobs1=n, alpha=alpha):.3f}')
need = tool.solve_power(d, power=0.80, alpha=alpha)
print(f'n per group for 80% power = {need:.1f}, so use {int(np.ceil(need))}')
rng = np.random.default_rng(3)
a = rng.normal(0, sd, (5000, n)); b = rng.normal(diff, sd, (5000, n))
p = stats.ttest_ind(a, b, axis=1).pvalue
print(f'simulated power = {(p <= alpha).mean():.3f}')

effect size d = 0.30
power with 40 per group = 0.263
n per group for 80% power = 175.4, so use 176
simulated power = 0.264


**Answer.** d = 0.30. With 40 per group the exact power is 0.263, so the test would miss a real 3-minute gain about 74% of the time, and the simulation (0.264) agrees. For 80% power the team needs 175.4, so 176 visitors per group. A small effect needs a large sample, so the plan is too weak as it stands.

## The same question in R, SPSS and Stata

**R**

```r
power.t.test(n = 36, delta = 6, sd = 15)                # power for 36 per group  # power.t.test computes power for a two-sample t-test: 36 per group, a real gap (delta) of 6 and an SD of 15. Leaving out power makes R solve for it.
power.t.test(power = 0.80, delta = 6, sd = 15)           # n per group for 80% power  # Leave out n and give power = 0.80: R solves for the group size needed.

# simulation (random, so close to but not exactly the Python value)  # A comment (R ignores it). The next lines are a simulation, so numbers will be close to Python's, not identical.
set.seed(42)  # Fix R's random numbers so the simulation repeats.
mean(replicate(10000, t.test(rnorm(36, 0, 15), rnorm(36, 0, 15))$p.value <= 0.05))  # Simulate 10,000 experiments with no real gap; t.test()$p.value <= 0.05 marks rejections and mean() gives the Type I error rate.
mean(replicate(10000, t.test(rnorm(36, 0, 15), rnorm(36, 6, 15))$p.value <= 0.05))  # Same simulation with a true gap of 6; the rejection rate is the power.
```

How to read it: power.t.test prints power = 0.3875 for 36 per group, and n = 99.08 per group for 80% power (use 100). The two simulated rates should be near 0.05 (false alarms) and near 0.39 (power), but not exactly the same as Python's.

**SPSS** (Analyze > Power Analysis > Means > Independent-Samples T Test (enter power 0.80 to solve for the sample size))

```
DATA LIST FREE / n delta sd.
BEGIN DATA
36 6 15
END DATA.
COMPUTE df = 2 * n - 2.
COMPUTE nc = (delta / sd) * SQRT(n / 2).
COMPUTE tcrit = IDF.T(0.975, df).
COMPUTE power = 1 - NCDF.T(tcrit, df, nc) + NCDF.T(-tcrit, df, nc).
LIST power.
```

How to read it: LIST shows power as about 0.39 (SPSS rounds the display; the exact value is 0.3875). That means a real 6-point gain would be detected only about 39% of the time with 36 per group. Use the Power Analysis menu to solve for the 100 per group needed.

**Stata**

```
power twomeans 0 6, sd(15) n1(36) n2(36)      // power for 36 per group
power twomeans 0 6, sd(15) power(0.80)         // sample size for 80% power
```

How to read it: The first table shows Estimated power = 0.3875 for N1 = N2 = 36. The second shows the sample size for 80% power, N1 = N2 = 100 per group (Stata rounds up from 99.08), matching the Python and R results.